In [ ]:
!git clone https://github.com/Atha1407/MediRent-Ai.git

In [ ]:
import os

print(os.listdir("/content"))

In [ ]:
os.chdir("/content/MediRent-Ai")

print("Current directory:", os.getcwd())

In [ ]:
import os

DATA_PATH = "data/processed/feature_engineered_medical_equipment_dataset.csv"

print("File exists:", os.path.exists(DATA_PATH))

In [ ]:
import pandas as pd

df = pd.read_csv(DATA_PATH)


In [ ]:
print("Dataset Shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())

In [ ]:
print("Number of columns:", len(df.columns))

print("\nColumns:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

In [ ]:
print(df.dtypes)

In [ ]:
missing = df.isnull().sum()

missing = missing[missing > 0].sort_values(ascending=False)

print("Columns with missing values:")
display(missing)

In [ ]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

In [ ]:
df["Booking_Date"] = pd.to_datetime(
    df["Booking_Date"],
    errors="coerce"
)

df["Rental_Start_Date"] = pd.to_datetime(
    df["Rental_Start_Date"],
    errors="coerce"
)

print(df[["Booking_Date", "Rental_Start_Date"]].dtypes)

In [ ]:
print("Missing Booking_Date:", df["Booking_Date"].isna().sum())
print("Missing Rental_Start_Date:", df["Rental_Start_Date"].isna().sum())

In [ ]:
print("Rental duration statistics:")
display(df["Rental_Duration_Days"].describe())

In [ ]:
invalid_duration = df[
    (df["Rental_Duration_Days"].isna()) |
    (df["Rental_Duration_Days"] <= 0)
]

print("Invalid rental duration records:", len(invalid_duration))

In [ ]:
print("Number of equipment types:",
      df["Equipment"].nunique())

print("\nEquipment types:")
print(sorted(df["Equipment"].unique()))

In [ ]:
# Cancelled rentals should not contribute to equipment utilisation

util_df = df[df["Rental_Status"] != "Cancelled"].copy()

print("Total records:", len(df))
print("Cancelled records:", (df["Rental_Status"] == "Cancelled").sum())
print("Records used for utilisation:", len(util_df))

In [ ]:
util_df["Rental_End_Date"] = (
    util_df["Rental_Start_Date"] +
    pd.to_timedelta(util_df["Rental_Duration_Days"], unit="D")
)

display(
    util_df[
        [
            "Equipment",
            "Rental_Start_Date",
            "Rental_Duration_Days",
            "Rental_End_Date"
        ]
    ].head()
)

In [ ]:
invalid_dates = util_df[
    util_df["Rental_End_Date"] <= util_df["Rental_Start_Date"]
]

print("Invalid rental date records:", len(invalid_dates))

In [ ]:
inventory_check = (
    util_df.groupby("Equipment")["Total_Inventory"]
    .nunique()
)

inconsistent_inventory = inventory_check[inventory_check > 1]

print("Equipment types with inconsistent inventory values:")
display(inconsistent_inventory)

print("\nNumber of inconsistent equipment types:",
      len(inconsistent_inventory))

In [ ]:
util_df["Availability_Check"] = (
    util_df["Available_Units_At_Start"] +
    util_df["Unavailable_Units_At_Start"]
    == util_df["Total_Inventory"]
)

print("Rows satisfying:")
print("Available + Unavailable = Total Inventory")

print(
    util_df["Availability_Check"].value_counts()
)

In [ ]:
util_df.drop(columns=["Availability_Check"], inplace=True)

In [ ]:
equipment_inventory = (
    util_df.groupby(
        ["Equipment", "Equipment_Category"],
        as_index=False
    )["Total_Inventory"]
    .first()
    .sort_values("Equipment")
)

display(equipment_inventory)

In [ ]:
ANALYSIS_START = util_df["Rental_Start_Date"].min().normalize()
ANALYSIS_END = pd.Timestamp("2026-07-31")

print("Analysis period:")
print("Start:", ANALYSIS_START.date())
print("End:", ANALYSIS_END.date())

ANALYSIS_DAYS = (ANALYSIS_END - ANALYSIS_START).days + 1

print("Number of analysis days:", ANALYSIS_DAYS)

In [ ]:
util_df["Utilisation_Start"] = util_df["Rental_Start_Date"].clip(
    lower=ANALYSIS_START,
    upper=ANALYSIS_END
)

util_df["Utilisation_End"] = util_df["Rental_End_Date"].clip(
    lower=ANALYSIS_START + pd.Timedelta(days=1),
    upper=ANALYSIS_END + pd.Timedelta(days=1)
)

display(
    util_df[
        [
            "Equipment",
            "Rental_Start_Date",
            "Rental_End_Date",
            "Utilisation_Start",
            "Utilisation_End"
        ]
    ].head()
)

In [ ]:
util_df["Rental_Days_Within_Analysis"] = (
    util_df["Utilisation_End"] -
    util_df["Utilisation_Start"]
).dt.days.clip(lower=0)

print(
    "Total rental days within analysis period:",
    util_df["Rental_Days_Within_Analysis"].sum()
)

In [ ]:
equipment_rental_days = (
    util_df.groupby("Equipment", as_index=False)
    ["Rental_Days_Within_Analysis"]
    .sum()
    .rename(
        columns={
            "Rental_Days_Within_Analysis":
            "Planned_Rental_Days"
        }
    )
)

display(equipment_rental_days)

In [ ]:
equipment_utilisation = equipment_inventory.merge(
    equipment_rental_days,
    on="Equipment",
    how="left"
)

equipment_utilisation["Planned_Rental_Days"] = (
    equipment_utilisation["Planned_Rental_Days"]
    .fillna(0)
)

equipment_utilisation["Available_Equipment_Days"] = (
    equipment_utilisation["Total_Inventory"] *
    ANALYSIS_DAYS
)

display(equipment_utilisation)

In [ ]:
equipment_utilisation["Utilisation_%"] = (
    equipment_utilisation["Planned_Rental_Days"] /
    equipment_utilisation["Available_Equipment_Days"]
) * 100

equipment_utilisation["Utilisation_%"] = (
    equipment_utilisation["Utilisation_%"].round(2)
)

display(
    equipment_utilisation.sort_values(
        "Utilisation_%",
        ascending=False
    )
)

In [ ]:
print("Utilisation Statistics")

print(
    "Average utilisation:",
    round(equipment_utilisation["Utilisation_%"].mean(), 2),
    "%"
)

print(
    "Minimum utilisation:",
    round(equipment_utilisation["Utilisation_%"].min(), 2),
    "%"
)

print(
    "Maximum utilisation:",
    round(equipment_utilisation["Utilisation_%"].max(), 2),
    "%"
)

In [ ]:
category_utilisation = (
    equipment_utilisation
    .groupby("Equipment_Category", as_index=False)
    .agg(
        Planned_Rental_Days=("Planned_Rental_Days", "sum"),
        Available_Equipment_Days=("Available_Equipment_Days", "sum"),
        Total_Inventory=("Total_Inventory", "sum")
    )
)

category_utilisation["Utilisation_%"] = (
    category_utilisation["Planned_Rental_Days"] /
    category_utilisation["Available_Equipment_Days"]
) * 100

category_utilisation["Utilisation_%"] = (
    category_utilisation["Utilisation_%"].round(2)
)

display(
    category_utilisation.sort_values(
        "Utilisation_%",
        ascending=False
    )
)

In [ ]:
print("Equipment Utilisation Distribution:")

display(
    equipment_utilisation["Utilisation_%"].describe()
)

In [ ]:
LOW_THRESHOLD = equipment_utilisation["Utilisation_%"].quantile(0.25)
HIGH_THRESHOLD = equipment_utilisation["Utilisation_%"].quantile(0.75)

print("Under-utilised threshold:",
      round(LOW_THRESHOLD, 2), "%")

print("Highly-utilised threshold:",
      round(HIGH_THRESHOLD, 2), "%")

In [ ]:
def classify_utilisation(value):
    if value >= HIGH_THRESHOLD:
        return "Highly Utilised"
    elif value <= LOW_THRESHOLD:
        return "Under Utilised"
    else:
        return "Moderately Utilised"


equipment_utilisation["Utilisation_Status"] = (
    equipment_utilisation["Utilisation_%"]
    .apply(classify_utilisation)
)

display(
    equipment_utilisation[
        [
            "Equipment",
            "Equipment_Category",
            "Utilisation_%",
            "Utilisation_Status"
        ]
    ].sort_values(
        "Utilisation_%",
        ascending=False
    )
)

In [ ]:
print(
    equipment_utilisation["Utilisation_Status"]
    .value_counts()
)

In [ ]:
highly_utilised = (
    equipment_utilisation[
        equipment_utilisation["Utilisation_Status"]
        == "Highly Utilised"
    ]
    .sort_values("Utilisation_%", ascending=False)
)

display(highly_utilised)

In [ ]:
under_utilised = (
    equipment_utilisation[
        equipment_utilisation["Utilisation_Status"]
        == "Under Utilised"
    ]
    .sort_values("Utilisation_%")
)

display(under_utilised)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plot_df = equipment_utilisation.sort_values(
    "Utilisation_%",
    ascending=True
)

plt.figure(figsize=(12, 8))

sns.barplot(
    data=plot_df,
    x="Utilisation_%",
    y="Equipment"
)

plt.axvline(
    LOW_THRESHOLD,
    linestyle="--",
    label=f"Under-utilised threshold ({LOW_THRESHOLD:.1f}%)"
)

plt.axvline(
    HIGH_THRESHOLD,
    linestyle="--",
    label=f"Highly-utilised threshold ({HIGH_THRESHOLD:.1f}%)"
)

plt.xlabel("Utilisation (%)")
plt.ylabel("Equipment")
plt.title("Equipment-wise Utilisation")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plot_category = category_utilisation.sort_values(
    "Utilisation_%",
    ascending=True
)

plt.figure(figsize=(10, 6))

sns.barplot(
    data=plot_category,
    x="Utilisation_%",
    y="Equipment_Category"
)

plt.xlabel("Utilisation (%)")
plt.ylabel("Equipment Category")
plt.title("Category-wise Equipment Utilisation")
plt.tight_layout()
plt.show()

In [ ]:
AS_OF_DATE = pd.Timestamp("2026-07-31")

print("Availability snapshot date:",
      AS_OF_DATE.date())

In [ ]:
current_rentals = util_df[
    (util_df["Rental_Start_Date"] <= AS_OF_DATE) &
    (util_df["Rental_End_Date"] > AS_OF_DATE)
].copy()

print(
    "Currently planned active rental records:",
    len(current_rentals)
)

display(
    current_rentals[
        [
            "Equipment",
            "Rental_Start_Date",
            "Rental_End_Date",
            "Rental_Duration_Days"
        ]
    ].head()
)

In [ ]:
current_rented = (
    current_rentals
    .groupby("Equipment")
    .size()
    .reset_index(name="Currently_Rented")
)

display(current_rented)

In [ ]:
current_availability = equipment_inventory.merge(
    current_rented,
    on="Equipment",
    how="left"
)

current_availability["Currently_Rented"] = (
    current_availability["Currently_Rented"]
    .fillna(0)
    .astype(int)
)

current_availability["Currently_Available"] = (
    current_availability["Total_Inventory"] -
    current_availability["Currently_Rented"]
)

display(
    current_availability.sort_values(
        "Currently_Available"
    )
)

In [ ]:
availability_valid = (
    current_availability["Currently_Available"] >= 0
).all()

print(
    "No equipment has negative availability:",
    availability_valid
)

print(
    "Total inventory:",
    current_availability["Total_Inventory"].sum()
)

print(
    "Currently rented:",
    current_availability["Currently_Rented"].sum()
)

print(
    "Currently available:",
    current_availability["Currently_Available"].sum()
)

In [ ]:
print(
    "Negative availability:",
    (current_availability["Currently_Available"] < 0).sum()
)

print(
    "Total inventory:",
    current_availability["Total_Inventory"].sum()
)

print(
    "Currently rented:",
    current_availability["Currently_Rented"].sum()
)

print(
    "Currently available:",
    current_availability["Currently_Available"].sum()
)

print(
    "\nCheck:",
    current_availability["Total_Inventory"].sum()
    ==
    current_availability["Currently_Rented"].sum()
    +
    current_availability["Currently_Available"].sum()
)

In [ ]:
equipment_utilisation_final = equipment_utilisation[
    [
        "Equipment",
        "Equipment_Category",
        "Total_Inventory",
        "Planned_Rental_Days",
        "Available_Equipment_Days",
        "Utilisation_%",
        "Utilisation_Status"
    ]
].copy()

display(
    equipment_utilisation_final.sort_values(
        "Utilisation_%",
        ascending=False
    )
)

In [ ]:
equipment_utilisation_final = equipment_utilisation_final.merge(
    current_availability[
        [
            "Equipment",
            "Currently_Rented",
            "Currently_Available"
        ]
    ],
    on="Equipment",
    how="left"
)

display(equipment_utilisation_final)

In [ ]:
def generate_insight(row):
    if row["Utilisation_Status"] == "Highly Utilised":
        return (
            f"{row['Equipment']} has high utilisation "
            f"({row['Utilisation_%']:.2f}%) and "
            f"{row['Currently_Available']} unit(s) currently available."
        )

    elif row["Utilisation_Status"] == "Under Utilised":
        return (
            f"{row['Equipment']} is under-utilised "
            f"({row['Utilisation_%']:.2f}%) with "
            f"{row['Currently_Available']} unit(s) currently available."
        )

    else:
        return (
            f"{row['Equipment']} has moderate utilisation "
            f"({row['Utilisation_%']:.2f}%) and "
            f"{row['Currently_Available']} unit(s) currently available."
        )


equipment_utilisation_final["Actionable_Insight"] = (
    equipment_utilisation_final.apply(
        generate_insight,
        axis=1
    )
)

display(
    equipment_utilisation_final[
        [
            "Equipment",
            "Utilisation_%",
            "Utilisation_Status",
            "Currently_Available",
            "Actionable_Insight"
        ]
    ]
)

In [ ]:
print("========== FINAL VALIDATION ==========")

print(
    "Utilisation below 0%:",
    (equipment_utilisation_final["Utilisation_%"] < 0).sum()
)

print(
    "Utilisation above 100%:",
    (equipment_utilisation_final["Utilisation_%"] > 100).sum()
)

print(
    "Negative available units:",
    (equipment_utilisation_final["Currently_Available"] < 0).sum()
)

print(
    "Missing utilisation values:",
    equipment_utilisation_final["Utilisation_%"].isna().sum()
)

print(
    "Missing equipment names:",
    equipment_utilisation_final["Equipment"].isna().sum()
)

In [ ]:
output_path = (
    "data/processed/equipment_utilisation.csv"
)

equipment_utilisation_final.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)

In [ ]:
category_output_path = (
    "data/processed/category_utilisation.csv"
)

category_utilisation.to_csv(
    category_output_path,
    index=False
)

print("Saved:", category_output_path)

In [ ]:
availability_output_path = (
    "data/processed/equipment_current_availability.csv"
)

current_availability.to_csv(
    availability_output_path,
    index=False
)

print("Saved:", availability_output_path)

In [ ]:
print("======================================")
print(" EQUIPMENT UTILISATION ANALYSIS DONE")
print("======================================")

print(f"Equipment types analysed: {len(equipment_utilisation_final)}")

print(
    f"Average utilisation: "
    f"{equipment_utilisation_final['Utilisation_%'].mean():.2f}%"
)

print(
    f"Highest utilisation: "
    f"{equipment_utilisation_final['Utilisation_%'].max():.2f}%"
)

print(
    f"Lowest utilisation: "
    f"{equipment_utilisation_final['Utilisation_%'].min():.2f}%"
)

print(
    f"Highly utilised equipment: "
    f"{(equipment_utilisation_final['Utilisation_Status'] == 'Highly Utilised').sum()}"
)

print(
    f"Under-utilised equipment: "
    f"{(equipment_utilisation_final['Utilisation_Status'] == 'Under Utilised').sum()}"
)

print(
    f"Currently rented units: "
    f"{equipment_utilisation_final['Currently_Rented'].sum()}"
)

print(
    f"Currently available units: "
    f"{equipment_utilisation_final['Currently_Available'].sum()}"
)

print("\nOutput files created:")
print("1. equipment_utilisation.csv")
print("2. category_utilisation.csv")
print("3. equipment_current_availability.csv")

### Utilisation interpretation

Because the dataset does not contain `Actual_Return_Date` or a physical `Equipment_ID`, the utilisation metric is interpreted as **planned-duration-based, equipment-type pool-level utilisation**. `Planned_Rental_Days` is derived from `Rental_Start_Date + Rental_Duration_Days`; it should not be described as actual physical-unit rental days.
